# Birkhoff--von Neumann cycle certificate

This notebook generates `fig:birkhoff-von-neumann-cycle`.  The figure illustrates the proof of the Birkhoff--von Neumann theorem: a bistochastic matrix which is not a permutation matrix has fractional support, and the corresponding bipartite graph contains an alternating cycle.  Perturbing the coupling along this minimal cycle preserves all row and column sums, so the point is not extreme.

In [1]:
from pathlib import Path
import os
import shutil
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, to_rgba
from matplotlib.patches import Circle, FancyArrowPatch, Rectangle

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

from figure_style import BLUE, RED, VIOLET, ORANGE, GRAY, figure_dir, remove_axes, save_pdf, setup_matplotlib

setup_matplotlib()

NAME = "birkhoff-von-neumann-cycle"
OUT = figure_dir(NAME)
ARXIV_OUT = Path.cwd() / "arxiv" / "figures"
if not ARXIV_OUT.exists():
    ARXIV_OUT = Path.cwd().parent / "arxiv" / "figures"
THUMB_OUT = Path.cwd() / "notebooks-figures" / "thumbnails"
if not THUMB_OUT.exists():
    THUMB_OUT = Path.cwd().parent / "notebooks-figures" / "thumbnails"
ARXIV_OUT.mkdir(parents=True, exist_ok=True)
THUMB_OUT.mkdir(parents=True, exist_ok=True)


## A bistochastic non-permutation matrix

The matrix has three isolated unit entries and one longer fractional alternating cycle.  The fractional rows and columns are deliberately scattered in the displayed order, so the non-binary part is not seen as a contiguous block.  The cycle is the obstruction to extremality: moving a small amount of mass with alternating signs along it preserves all row and column sums.


In [2]:
P = np.zeros((7, 7))

# A scattered 8-edge fractional cycle using rows 1,3,6,7 and columns 4,1,7,2
# in human indexing.  This avoids the visual artifact of a contiguous fractional block.
a = 0.37
cycle_values = {
    (0, 3): a,       (0, 0): 1 - a,
    (2, 0): a,       (2, 6): 1 - a,
    (5, 6): a,       (5, 1): 1 - a,
    (6, 1): a,       (6, 3): 1 - a,
}
for (i, j), val in cycle_values.items():
    P[i, j] = val

# Three isolated unit entries complete the bistochastic matrix.
for i, j in [(1, 5), (3, 2), (4, 4)]:
    P[i, j] = 1.0

assert np.allclose(P.sum(axis=0), 1)
assert np.allclose(P.sum(axis=1), 1)
assert not np.all((P == 0) | (P == 1))

cycle_order = [(0, 3), (0, 0), (2, 0), (2, 6), (5, 6), (5, 1), (6, 1), (6, 3)]
cycle_edges = set(cycle_order)
unit_edges = {(i, j) for i in range(7) for j in range(7) if np.isclose(P[i, j], 1.0)}
fractional_edges = {(i, j) for i in range(7) for j in range(7) if 1e-12 < P[i, j] < 1 - 1e-12}
assert len(unit_edges) == 3
assert fractional_edges == cycle_edges
print("row sums", P.sum(axis=1))
print("column sums", P.sum(axis=0))
print("unit edges", sorted(unit_edges))
print("fractional cycle", cycle_order)


row sums [1. 1. 1. 1. 1. 1. 1.]
column sums [1. 1. 1. 1. 1. 1. 1.]
unit edges [(1, 5), (3, 2), (4, 4)]
fractional cycle [(0, 3), (0, 0), (2, 0), (2, 6), (5, 6), (5, 1), (6, 1), (6, 3)]


## Matrix and support graph

In the matrix panel, the eight fractional entries are colored in violet and the three unit entries in dark gray.  In the graph panel, red nodes are rows and blue nodes are columns.  Thin purple edges correspond to $0<P_{ij}<1$, bold dark edges to $P_{ij}=1$, and the orange halo marks the longer alternating fractional cycle.  The vertical stack is intentionally compact so that the two panels align cleanly in the book.


In [3]:
def draw_matrix_panel(path):
    fig, ax = plt.subplots(figsize=(1.72, 1.72))
    ax.set_xlim(-0.5, 6.5)
    ax.set_ylim(6.5, -0.5)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(0.68)
        spine.set_color("#303030")

    # Cell backgrounds: white for zero, violet for fractional, dark gray for one.
    frac_cmap = LinearSegmentedColormap.from_list("frac", ["#f8f4fb", "#c79bd3", VIOLET])
    for i in range(7):
        for j in range(7):
            val = P[i, j]
            if val == 0:
                color = "#ffffff"
            elif val < 1:
                color = frac_cmap(0.25 + 0.65 * val)
            else:
                color = "#202020"
            ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor=color, edgecolor="none"))

    # Highlight the selected alternating cycle cell by cell, since it is no longer a block.
    for i, j in cycle_order:
        ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor=ORANGE, linewidth=1.18, zorder=5))

    # Grid and values.
    for k in np.arange(-0.5, 7.5, 1):
        ax.axhline(k, color="#2d2d2d", lw=0.34, zorder=3)
        ax.axvline(k, color="#2d2d2d", lw=0.34, zorder=3)
    for i in range(7):
        for j in range(7):
            val = P[i, j]
            if val > 0:
                text = "1" if np.isclose(val, 1) else f"{val:.2f}"
                color = "white" if np.isclose(val, 1) else "#231026"
                ax.text(j, i, text, ha="center", va="center", fontsize=7.2, color=color, zorder=6)

    # Row/column labels are kept close to the matrix to reduce the total footprint.
    for i in range(7):
        ax.text(-0.76, i, rf"$i_{i+1}$", ha="center", va="center", fontsize=7.2, color=RED)
        ax.text(i, -0.76, rf"$j_{i+1}$", ha="center", va="center", fontsize=7.2, color=BLUE)
    save_pdf(fig, path, pad_inches=0.02)
    plt.close(fig)


def draw_graph_panel(path):
    fig, ax = plt.subplots(figsize=(2.05, 1.72))
    remove_axes(ax)
    ax.set_aspect("equal")

    # Compact vertical stack.  Columns are permuted for readability while the
    # matrix itself keeps the scattered index pattern.
    y_vals = np.linspace(1.62, -1.62, 7)
    y_left = {i: y_vals[i] for i in range(7)}
    y_right = {
        3: y_left[0],  # fractional cycle
        0: y_left[2],
        6: y_left[5],
        1: y_left[6],
        5: y_left[1],  # isolated unit entries
        2: y_left[3],
        4: y_left[4],
    }
    x_left, x_right = -1.14, 1.14
    pos_left = {i: np.array([x_left, y_left[i]]) for i in range(7)}
    pos_right = {j: np.array([x_right, y_right[j]]) for j in range(7)}

    def draw_edge(i, j, *, color, lw, alpha=1.0, zorder=1, rad=0.0):
        p, q = pos_left[i], pos_right[j]
        patch = FancyArrowPatch(
            p, q,
            arrowstyle="-",
            connectionstyle=f"arc3,rad={rad}",
            linewidth=lw,
            color=color,
            alpha=alpha,
            zorder=zorder,
            capstyle="round",
            joinstyle="round",
        )
        ax.add_patch(patch)

    # Unit edges are isolated in bold dark strokes.
    for i, j in sorted(unit_edges):
        draw_edge(i, j, color="#242424", lw=1.65, alpha=0.94, zorder=1)

    # A soft orange halo marks the selected cycle, while the support edges
    # themselves remain thin purple strokes.
    rad_by_edge = {
        (0, 3): 0.00, (0, 0): -0.08,
        (2, 0): 0.00, (2, 6): -0.06,
        (5, 6): 0.00, (5, 1): -0.06,
        (6, 1): 0.00, (6, 3): 0.14,
    }
    for i, j in cycle_order:
        draw_edge(i, j, color=ORANGE, lw=2.35, alpha=0.32, zorder=2, rad=rad_by_edge[(i, j)])
    for i, j in cycle_order:
        draw_edge(i, j, color=VIOLET, lw=0.84, alpha=0.82, zorder=3, rad=rad_by_edge[(i, j)])

    # Nodes and labels.
    for i in range(7):
        ax.add_patch(Circle(pos_left[i], 0.065, facecolor=RED, edgecolor="white", linewidth=0.48, zorder=5))
        ax.text(x_left - 0.16, y_left[i], rf"$i_{i+1}$", ha="right", va="center", fontsize=7.3, color=RED)
    for j in range(7):
        ax.add_patch(Circle(pos_right[j], 0.065, facecolor=BLUE, edgecolor="white", linewidth=0.48, zorder=5))
        ax.text(x_right + 0.16, y_right[j], rf"$j_{j+1}$", ha="left", va="center", fontsize=7.3, color=BLUE)

    # A compact cyclic arrow suggests the alternating perturbation direction.
    center = np.array([0.0, -0.05])
    theta = np.linspace(0.15 * np.pi, 1.85 * np.pi, 80)
    rx, ry = 0.25, 0.25
    ax.plot(center[0] + rx * np.cos(theta), center[1] + ry * np.sin(theta), color=ORANGE, lw=0.72, alpha=0.72, zorder=4)
    ax.annotate("", xy=(center[0] + rx * np.cos(theta[-1]), center[1] + ry * np.sin(theta[-1])),
                xytext=(center[0] + rx * np.cos(theta[-3]), center[1] + ry * np.sin(theta[-3])),
                arrowprops=dict(arrowstyle="-|>", lw=0.68, color=ORANGE, mutation_scale=5.4))

    ax.set_xlim(-1.55, 1.55)
    ax.set_ylim(-1.92, 1.92)
    save_pdf(fig, path, pad_inches=0.018)
    plt.close(fig)


def copy_to_arxiv(pdf_name):
    shutil.copyfile(OUT / pdf_name, ARXIV_OUT / f"{NAME}--{pdf_name}")


draw_matrix_panel(OUT / "matrix.pdf")
draw_graph_panel(OUT / "support-graph.pdf")
copy_to_arxiv("matrix.pdf")
copy_to_arxiv("support-graph.pdf")


'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


'created' timestamp seems very low; regarding as unix timestamp


'modified' timestamp seems very low; regarding as unix timestamp


In [4]:
# Thumbnail for the notebook gallery.
# Build a PNG thumbnail from the freshly rendered vector panels.  Prefer
# PyMuPDF when available, and otherwise use the same Poppler renderer as the
# visual QA workflow.
from PIL import Image
import subprocess
import tempfile


def render_pdf_to_image(pdf, scale=3.0):
    try:
        import fitz
        doc = fitz.open(pdf)
        page = doc[0]
        pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
        im = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        doc.close()
        return im
    except Exception:
        with tempfile.TemporaryDirectory() as tmp:
            prefix = Path(tmp) / "panel"
            subprocess.run(
                ["pdftoppm", "-png", "-r", str(int(72 * scale)), str(pdf), str(prefix)],
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            return Image.open(f"{prefix}-1.png").convert("RGB")

imgs = [render_pdf_to_image(OUT / "matrix.pdf"), render_pdf_to_image(OUT / "support-graph.pdf")]
h = max(im.height for im in imgs)
pad = 22
canvas = Image.new("RGB", (sum(im.width for im in imgs) + 3 * pad, h + 2 * pad), "white")
x0 = pad
for im in imgs:
    canvas.paste(im, (x0, pad + (h - im.height) // 2))
    x0 += im.width + pad
thumb = THUMB_OUT / f"{NAME}.png"
canvas.save(thumb, quality=92)
print(thumb)


/Users/gpeyre/Dropbox/github/ot4ml/notebooks-figures/thumbnails/birkhoff-von-neumann-cycle.png
